# 01 · First-gift cohorts, the second-gift label, and who is actually in the data

**Workstream: data and label construction — Bakul.** Everything downstream depends on this
notebook, so it is written to be read aloud, not skimmed.

The project asks one question: *a donor gave to a classroom for the first time — will they give
again within a year, and should our stakeholder spend one of her scarce follow-up slots on them?*

Turning that sentence into a column is most of the work, and it is where the project can quietly
die. This notebook does five things:

1. Confirms the donor id actually links gifts across projects — Albert's blocking check.
2. Builds the first-gift cohorts and the label, with every judgement call stated.
3. Cuts the time-based split.
4. Runs the honest baseline so every later improvement has something to be measured against.
5. **Finds that the file holds three populations, not one — and that pooling them would have
   made every result in this project wrong.** That is the section to read if you read only one.

The logic lives in `src/`, not in these cells, because four other notebooks import it and a
definition that exists in two places will drift. Data: ICPSR 37898, `DS0001` donations,
11,377,479 rows. Runs in about ten minutes on a laptop.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src")); sys.path.insert(0, str(REPO / "evals"))

import numpy as np
import pandas as pd
import config, labels, baselines, score

pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

DONATIONS = REPO / "data" / "raw" / "ICPSR_37898" / "DS0001" / "37898-0001-Data.tsv"
assert DONATIONS.exists(), f"Download ICPSR_37898-V1.zip and unzip into data/raw/ — see data/README.md. Missing: {DONATIONS}"
print(f"Source: {DONATIONS.name}  ({DONATIONS.stat().st_size / 1e9:.2f} GB)")

Source: 37898-0001-Data.tsv  (1.57 GB)


## 1 · The check that gates everything

Albert, approving the proposal:

> *"As soon as possible, confirm that the donor ID in the public-use file actually links gifts
> across projects, because the whole project depends on it."*

He is right, and it is worth being explicit about why. Released datasets often mint a fresh
identifier per record to protect privacy. If DonorsChoose had done that here, every donor would
appear exactly once, every donor would look like a one-time giver **by construction**, our label
would be all zeros, and there would be no project — not because donors do not come back, but
because the file cannot see them coming back.

The test is not "do ids repeat" but "do ids repeat **across different projects**". Repeats within a
single project could just be an id scoped to that project.

In [2]:
donations = labels.load_donations(DONATIONS, verbose=False)
print(f"{len(donations):,} donation rows, {donations['donor_id'].nunique():,} distinct donors")

per_donor = donations.groupby("donor_id").agg(gifts=("amount", "size"), projects=("project_id", "nunique"))
multi = (per_donor["projects"] > 1).mean()
once  = (per_donor["gifts"] == 1).mean()
print(f"Donors giving to more than one distinct project : {multi:.1%}   ← the answer")
print(f"Donors who gave exactly once, ever              : {once:.1%}   ← the problem statement, with a number on it")
print("PASS — ids follow the person." if multi > 0.05 else "STOP — ids do not link. Switch to the fallback.")

11,377,479 donation rows, 3,466,570 distinct donors


Donors giving to more than one distinct project : 25.4%   ← the answer
Donors who gave exactly once, ever              : 71.1%   ← the problem statement, with a number on it
PASS — ids follow the person.


**It passes.** A quarter of donors give to more than one distinct project, so the id follows the
person. And the same query hands us the opening line of the presentation: across seventeen years,
**71% of DonorsChoose donors gave exactly once and never came back.** Malorie's proposal said
"most nonprofits acquire a donor once and never hear from them again." That is now a measurement.

## 2 · Building the label, and the four calls it rests on

`labels.build_cohorts()` reduces the donation rows to one row per donor. Four decisions are baked
in, and each could defensibly have gone the other way — so each is stated here rather than buried.

**(a) Same-month repeats do not count as a second gift.**
DonorsChoose lets a donor fund several classrooms in a single checkout, and this release records
month, not day. So two donations in the donor's first month are more plausibly *one* giving event
split across classrooms than a genuine return visit. Counting them would inflate the positive class
with something our stakeholder cannot act on — she cannot steward back a donor who never left. The
diagnostics below report the rate both ways so the choice is visible, not assumed.

**(b) The window is twelve whole calendar months, M+1 to M+12.** No day precision exists.

**(c) Cohorts whose window has not closed are dropped, not labelled zero.**
A donor whose first gift is in mid-2019 has not had a year to come back — the data simply stops.
Labelling them 0 would teach the model that recent donors do not return, which is an artefact of
the cutoff rather than a fact about donors.

**(d) Refunds are not gifts.** The codebook lists a minimum amount of −15.00. A reversal cannot
open a cohort or count as a return. There are 151 such rows; they are excluded, not zeroed.

In [3]:
cohorts, diagnostics = labels.build_cohorts(donations, verbose=False)
for k, v in diagnostics.items():
    print(f"  {k:<40} {v:,.4f}" if isinstance(v, float) else f"  {k:<40} {v:,}")
print()
print(cohorts.groupby("split").agg(donors=("donor_id", "size"), positive_rate=("gave_again", "mean")))

  rows_dropped_nonpositive_amount          151
  donors_before_window_filter              3,466,533
  positive_rate_strict                     0.1525
  share_with_multiple_first_month_gifts    0.1047
  donors_dropped_unclosed_window           189,083
  donors_dropped_pre_coverage              297
  donors_labelled                          3,277,153
  positive_rate                            0.1579

          donors  positive_rate
split                          
holdout   943137          0.161
train    2334016          0.157


In [4]:
# Decision (a), quantified: how much would the positive rate move if same-month repeats counted?
loose, loose_diag = labels.build_cohorts(donations, same_month_counts=True, verbose=False)
print(f"Positive rate, same-month repeats EXCLUDED (our choice) : {diagnostics['positive_rate']:.4f}")
print(f"Positive rate, same-month repeats INCLUDED               : {loose_diag['positive_rate']:.4f}")
print(f"Share of donors with >1 gift in their first month        : {diagnostics['share_with_multiple_first_month_gifts']:.1%}")
del loose

Positive rate, same-month repeats EXCLUDED (our choice) : 0.1579
Positive rate, same-month repeats INCLUDED               : 0.2703
Share of donors with >1 gift in their first month        : 10.5%


## 3 · The split — time-based, never random

Train on first-gift cohorts through 2016; hold out 2017 and 2018.

A random split would be indefensible here. Our stakeholder's question is about *next year's* new
donors, so the honest test is whether a model fitted on the past predicts a future it has not seen.
A random split would let it learn from 2018 donors to predict 2017 ones, which is not a situation
she will ever be in, and would flatter the result.

The holdout is not looked at again until a result is final. Below is the one thing we do look at:
whether the repeat rate drifts across years, because if it does, a model trained on old cohorts
will be miscalibrated on new ones — a finding for the error analysis, not a bug.

In [5]:
by_year = (cohorts.assign(year=cohorts["cohort"].str[:4])
           .groupby("year").agg(donors=("donor_id", "size"), repeat_rate=("gave_again", "mean"),
                                median_first_gift=("first_gift_amount", "median")))
print(by_year.to_string())

      donors  repeat_rate  median_first_gift
year                                        
2003    1687        0.200            200.000
2004    2327        0.239            110.000
2005    3411        0.270            117.000
2006   12691        0.179             80.000
2007   36279        0.296             50.000
2008   51657        0.295             35.000
2009   76868        0.240             25.000
2010  204793        0.153             20.000
2011  317742        0.120             25.000
2012  294535        0.110             20.000
2013  267123        0.167             25.000
2014  291914        0.174             26.365
2015  339022        0.156             35.290
2016  433967        0.154             40.000
2017  427343        0.172             50.000
2018  515794        0.152             46.110


## 4 · Who is actually in this file — the finding that changes everything downstream

The first time the scorer ran on real data it reported that ranking donors by first-gift size
found **80% of all subsequent giving at 10% capacity.** That looked like a triumph. It was an
artefact, and it is worth walking through exactly how, because it is the most useful thing in this
notebook.

The donations file carries a `DONOR_TYPE` column with three values. They are not one population:

In [6]:
h = cohorts[cohorts["split"] == "holdout"]
g = h.groupby("donor_type").agg(donors=("donor_id", "size"), repeat_rate=("gave_again", "mean"),
                                median_first_gift=("first_gift_amount", "median"),
                                subsequent_value=("second_gift_amount", "sum"))
g["donor_share"] = g["donors"] / g["donors"].sum()
g["value_share"] = g["subsequent_value"] / g["subsequent_value"].sum()
print(g[["donors", "donor_share", "repeat_rate", "median_first_gift", "subsequent_value", "value_share"]].to_string())

print("\nThe five largest second-gift totals in the holdout:")
print(h.nlargest(5, "second_gift_amount")[["donor_type", "cohort", "first_gift_amount", "second_gift_count", "second_gift_amount"]].to_string(index=False))

               donors  donor_share  repeat_rate  median_first_gift  subsequent_value  value_share
donor_type                                                                                       
citizen donor  814971        0.864        0.126             40.000    16,792,616.780        0.231
organization      708        0.001        0.698         11,988.195    45,405,467.210        0.625
teacher        127458        0.135        0.382             65.000    10,472,742.800        0.144

The five largest second-gift totals in the holdout:
  donor_type  cohort  first_gift_amount  second_gift_count  second_gift_amount
organization 2018-04        217,069.090              66348       2,251,041.350
organization 2017-06         29,601.550              36426       1,995,481.080
organization 2017-10      4,090,076.520               2271       1,808,187.280
organization 2018-08         44,459.430              47833       1,770,072.440
organization 2017-04        595,577.650              25112    

Read the `organization` row. Seven hundred of them — one tenth of one percent of donors — hold
**62% of every dollar that came back.** The largest made tens of thousands of donations inside its
twelve-month window. That is a corporate matching program or a foundation running a campaign, not
a person a development lead calls. Their first gifts are also enormous, so "rank by gift size"
finds them trivially. Any ranker, however dumb, wins the pooled contest by locating the
corporations — and the stakeholder already knows who her corporations are.

Teachers are a third population again: they give to their own classrooms, repeat at three times
the citizen rate, and are not stewardship targets either.

**The decision:** score citizen donors only. It lives in `config.STAKEHOLDER_POPULATION` as a
single line, the label is still built for everyone, and the pooled number is kept below for
contrast — so the choice is reversible and both results stay reproducible. It is a scoping call
for the team to confirm, and it is written into the config with the table above beside it.

## 5 · The honest baseline, on the right population

Small nonprofits rank donors by **RFM** — recency, frequency, monetary value. Applied to a cohort of
*first-time* donors it collapses: frequency is 1 for everyone by construction, recency is identical
within a monthly cohort, and only monetary value survives. So the strongest honest simple rule is
*rank by first gift size*, which is genuinely what a development lead does when she builds the list
by hand. Albert's condition on accepting a negative result was that the baselines be **fair**. This
one is the real practice, correctly named.

The scorer measures at her **capacity** — the top slice of each month's new donors she can reach —
because that is the decision. Not PR-AUC. And one claim we do not make: nobody here was randomly
assigned to be contacted, so we cannot measure *uplift from outreach*. We rank by predicted future
value and report value **identified**, never "retained" or "caused".

In [7]:
citizens, pop_label = score.select_population(cohorts)
hold = citizens[citizens["split"] == "holdout"].reset_index(drop=True)
print(f"Population: {pop_label} — {len(hold):,} holdout donors, {hold['gave_again'].mean():.1%} gave again, "
      f"${hold['second_gift_amount'].sum():,.0f} subsequent giving\n")
table = score.compare(hold, baselines.score_all_baselines(hold))
ref = score.contact_everyone_reference(hold)
print(f"Contact everyone: ${ref['value_per_contact']:,.2f} per contact\n")
print(table[table["capacity"] == config.STEWARDSHIP_CAPACITY][
    ["ranking", "donors_contacted", "precision", "recall", "value_capture_rate", "value_per_contact"]
].to_string(index=False))

Population: citizen donor — 814,971 holdout donors, 12.6% gave again, $16,792,617 subsequent giving



Contact everyone: $20.61 per contact

               ranking  donors_contacted  precision  recall  value_capture_rate  value_per_contact
           gift_amount             81509      0.197   0.157               0.564            116.258
first_month_gift_count             81509      0.204   0.162               0.414             85.304
                random             81509      0.127   0.101               0.091             18.665


In [8]:
# For contrast: the same measurement pooled across all three donor types.
# This is the number that looked like a triumph. It is measuring corporate matching programs.
hold_all = cohorts[cohorts["split"] == "holdout"].reset_index(drop=True)
t_all = score.compare(hold_all, {"gift_amount": baselines.rank_by_gift_amount(hold_all)}, [config.STEWARDSHIP_CAPACITY])
print(f"POOLED, all donor types, 10% capacity, rank by gift size: "
      f"{t_all.iloc[0]['value_capture_rate']:.1%} of value identified, ${t_all.iloc[0]['value_per_contact']:,.0f} per contact")
print(f"CITIZEN DONORS ONLY, same rule:                            "
      f"{table[(table.ranking=='gift_amount') & (table.capacity==config.STEWARDSHIP_CAPACITY)].iloc[0]['value_capture_rate']:.1%} of value identified")

POOLED, all donor types, 10% capacity, rank by gift size: 79.8% of value identified, $615 per contact
CITIZEN DONORS ONLY, same rule:                            56.4% of value identified


In [9]:
# Sanity-check the scorer before trusting any of it: a random ranking at capacity c must capture
# about c of the total value. If this does not hold, the ranking or masking is wrong.
for c in config.CAPACITY_SWEEP:
    row = score.evaluate_at_capacity(hold, baselines.rank_random(hold), c)
    print(f"  capacity {c:>5.0%}  →  random captures {row['value_capture_rate']:6.1%}  (should be ≈ {c:.0%})")

  capacity    1%  →  random captures   0.8%  (should be ≈ 1%)


  capacity    5%  →  random captures   4.3%  (should be ≈ 5%)


  capacity   10%  →  random captures   9.1%  (should be ≈ 10%)


  capacity   20%  →  random captures  19.2%  (should be ≈ 20%)


  capacity   50%  →  random captures  47.2%  (should be ≈ 50%)


## 6 · What the first real number says, and what it does not

At the stakeholder's capacity, on the population she actually serves, ranking by first-gift size
identifies a little over half of all subsequent giving, at roughly five times the dollars per
contact of working the whole list. That is the bar every model has to clear, and it is not a low
one.

But look at precision: **four out of five people on that list do not come back.** The baseline is
good at finding dollars and poor at finding people. Breadth of first gift — how many classrooms
someone funded in their first checkout — is slightly *better* at predicting who returns, while size
is better at predicting how much. Those are different questions, and which one the stakeholder is
asking is the economics workstream's to settle.

## 7 · Handoff

`data/processed/cohorts.parquet` — one row per donor, labelled, split, with first-gift attributes
and donor type — is the input to all four remaining workstreams.

| Workstream | Owner | Starts from |
|---|---|---|
| Exploratory analysis and features | Reid | `cohorts.parquet` joined to `DS0003` projects on `first_project_id` |
| Baseline and model development | Rodolfo | `cohorts.parquet`, train split, citizen donors; score the holdout once |
| Evaluation and error analysis | Thadeus | `evals/score.py`, then calibration and cuts by cohort year |
| Outreach economics and recommendations | Malorie | the capacity table, plus a real cost per contact |

**Open questions this notebook could not settle:**

1. The thank-you-packet flag: the codebook gives no timing. If the packet can be mailed *after* a
   second gift it leaks the answer. Albert said to drop that question without regret if ambiguous.
   It is ambiguous. Recommend dropping it as a feature; keep it as a descriptive statistic.
2. One `citizen donor` in the holdout made 1,424 second gifts totalling $884K. Either a
   misclassified organisation or a very unusual person. Worth a look before modelling.
3. Project text is masked by ICPSR — `ESSAY_TEXT`, `NEED_STATEMENT` all read `MASKED BY ICPSR`.
   Feature work is structured fields only; no embeddings on this dataset.